In [11]:
import os
from difflib import SequenceMatcher
from pathlib import Path

import cv2
import easyocr
import matplotlib.pyplot as plt
import numpy as np
import torch
from qwen_vl_utils import process_vision_info
from transformers import (AutoProcessor, AutoTokenizer,
                          Qwen2VLForConditionalGeneration)

In [16]:
class RecognizeModel:
    languages = {
        'ru': 'Russian',
        'en': 'English',
    }
    
    def __init__(self):
        self._model = Qwen2VLForConditionalGeneration.from_pretrained(
            "prithivMLmods/Qwen2-VL-OCR2-2B-Instruct",
            dtype=torch.bfloat16,
            device_map="auto",
        )
        self._processor = AutoProcessor.from_pretrained("prithivMLmods/Qwen2-VL-OCR-2B-Instruct")

        self._device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

    @classmethod
    def _get_prompt(cls, img_path, lang):
        prompt = f"Give me text from image, written in {cls.languages[lang]} language, nothing else."
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": img_path,
                    },
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        
        return messages

    def recognize(self, img_path, lang):
        img_path = str(img_path)
        
        messages = self._get_prompt(img_path, lang)
        text = self._processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self._processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        
        inputs = inputs.to(self._device_type)
        
        generated_ids = self._model.generate(**inputs, max_new_tokens=128)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = self._processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        
        return output_text[0].strip()
        

In [8]:
model = RecognizeModel()

Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [ ]:
reader = easyocr.Reader(["ru"], gpu=True)
images_dir = Path(r"C:\Projects\data-analysis\Титры Лунтика Новый 8K Смотреть На ПК и не только")

for filename in sorted(images_dir.rglob('*'), key=lambda x: (len(str(x)), str(x))):
    if filename.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
        continue

    print(f"Обработка: {filename.name}")
    
    # после перезапуска убраит str
    text = model.recognize(str(filename), 'ru')

    print(text)
    
    # with open(os.path.join(images_dir, filename), 'rb') as f:
    #     img = np.asarray(bytearray(f.read()), dtype=np.uint8)
    #     img = cv2.imdecode(img, cv2.IMREAD_COLOR)

    # if img is not None:
    #     img = extract_subtitle_region(img)
    #     text: str = recognize_text(img, reader)
        
    #     if text.strip():
    #         print(f"Субтитры: {text}")
    #     else:
    #         print("Субтитры не найдены")
    # else:
    #     print('Ошибка загрузки изображения')

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Обработка: frame_1.jpg
